In [36]:
import numpy as np
import pandas as pd
import string
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import MultinomialNB

In [37]:
df = pd.read_csv("Turkish_Book_Dataset_Kaggle_V2.csv", dtype=str)
print(df.shape)
df.head()

(67955, 8)


,book_title,book_publisher,book_author,book_category_name,book_productcode,book_page_count,book_released_year,book_detail
0,Klinik Uygulamada Bilişsel Davranışçı Terapi - 2,Nobel Akademik Yayıncılık,Mehmet Ak,academic,9786254272936,260,2022,Bilişsel davranışçı terapi (BDT) alanında çalı...
1,Taşra Üniversiteleri,İletişim Yayınevi,Tuğba Tekerek,academic,9789750534218,400,2023,"Tugˆba Tekerek, tas¸ra u¨niversitelerinin akad..."
2,Çocuklar ve Ergenler İçin Bilişsel Davranışçı ...,Nobel Akademik Yayıncılık,Arzu Aydın,academic,9786050330076,304,2021,Çocuklar ve Ergenler İçin Bilişsel-davranışçı ...
3,İki Dirhem Bir Çekirdek,Kapı Yayınları,İskender Pala,academic,9789758950379,212,2022,"Anlatımı güzelleştirmek, savunulan fikir ve dü..."
4,Reader at Work 2,ODTÜ Geliştirme Vakfı Yayıncılık,Kolektif,academic,9789754293999,550,2020,This book is the second volume of a collection...


In [38]:
print(df.isna().sum())

df_clean = df.dropna(subset=["book_detail", "book_category_name", "book_author"]).copy()
df_clean = df_clean[df_clean.book_detail.str.len() >= 50]
df_clean = df_clean.drop_duplicates("book_detail")
print("Temizlik sonrası kitap sayısı:", len(df_clean))

book_title               0
book_publisher         628
book_author           1154
book_category_name     628
book_productcode       628
book_page_count        712
book_released_year    6589
book_detail           2700
dtype: int64
Temizlik sonrası kitap sayısı: 59768


In [39]:
df_clean.book_category_name.value_counts()

book_category_name
literature                 28567
academic                   16293
business_and_economy        7601
science_and_engineering     3761
psychology                  3546
Name: count, dtype: int64

In [40]:
tablo = str.maketrans("", "", string.punctuation + "…“”‘’")

def clean(text):
    text = text.replace("I", "ı").replace("İ", "i").lower()
    return " ".join(text.translate(tablo).split())

df_clean["clean"] = df_clean.book_detail.apply(clean)
df_clean[["book_category_name", "clean"]].head()

,book_category_name,clean
0,academic,bilişsel davranışçı terapi bdt alanında çalışa...
1,academic,tugˆba tekerek tas¸ra u¨niversitelerinin akade...
2,academic,çocuklar ve ergenler için bilişseldavranışçı t...
3,academic,anlatımı güzelleştirmek savunulan fikir ve düş...
4,academic,this book is the second volume of a collection...


In [41]:
X = df_clean.clean.to_numpy()
y = df_clean.book_category_name.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), len(X_test))

47814 11954


In [42]:
vectorizer = CountVectorizer(min_df=2)
Xv_train = vectorizer.fit_transform(X_train)
Xv_test = vectorizer.transform(X_test)
len(vectorizer.vocabulary_)

163717

In [43]:
model = MultinomialNB()
model.fit(Xv_train, y_train)

pred_train = model.predict(Xv_train)
pred_test = model.predict(Xv_test)
print("Train accuracy:", accuracy_score(y_train, pred_train))
print("Test accuracy :", accuracy_score(y_test, pred_test))
print("Train macro F1:", f1_score(y_train, pred_train, average="macro"))
print("Test macro F1 :", f1_score(y_test, pred_test, average="macro"))

Train accuracy: 0.8762287196218681
Test accuracy : 0.8192236908147901
Train macro F1: 0.8188669130949698
Test macro F1 : 0.7143138177521935


In [44]:
print(classification_report(y_test, pred_test))

                         precision    recall  f1-score   support

               academic       0.73      0.78      0.76      3259
   business_and_economy       0.68      0.70      0.69      1520
             literature       0.94      0.95      0.94      5714
             psychology       0.73      0.66      0.69       709
science_and_engineering       0.60      0.42      0.49       752

               accuracy                           0.82     11954
              macro avg       0.74      0.70      0.71     11954
           weighted avg       0.82      0.82      0.82     11954



In [45]:
kelimeler = vectorizer.get_feature_names_out()
lp = model.feature_log_prob_
for i, sinif in enumerate(model.classes_):
    diger = np.delete(lp, i, axis=0).mean(axis=0)
    en_iyi = np.argsort(lp[i] - diger)[::-1][:10]
    print(f"{sinif:25s}", ", ".join(kelimeler[en_iyi]))

academic                  yargıtay, kanununun, muhakemesi, hukukunun, sayımızda, hukuku, milletlerarası, hükümleri, mahkemesinin, hukukunda
business_and_economy      işletmelerin, tablolari, pazarlamanın, finansal, muhasebe, pazarlama, bankacılık, döviz, eticaret, bitcoin
literature                xwe, ji, min, jî, wan, wê, wî, ser, şiirleri, ez
psychology                psikanaliz, psikoterapi, yalom, freudun, psikanalizin, freud, terapinin, terapi, çocuğunuzun, psikanalist
science_and_engineering   diyabetik, feynmanın, dawkins, fizikçi, hawkingin, seçilim, retinopati, hawking, evrimin, görelilik


In [46]:
def kategori_tahmin(metin):
    vec = vectorizer.transform([clean(metin)])
    olasiliklar = model.predict_proba(vec)[0]
    return dict(zip(model.classes_, olasiliklar.round(3)))

kategori_tahmin("Stres, kaygı ve depresyonla başa çıkmak için bilişsel davranışçı terapi teknikleri")

{np.str_('academic'): np.float64(0.0),
 np.str_('business_and_economy'): np.float64(0.0),
 np.str_('literature'): np.float64(0.0),
 np.str_('psychology'): np.float64(1.0),
 np.str_('science_and_engineering'): np.float64(0.0)}

In [47]:
kategori_tahmin("Aşk, ayrılık ve hüzün dolu, İstanbul'da geçen etkileyici bir roman")

{np.str_('academic'): np.float64(0.0),
 np.str_('business_and_economy'): np.float64(0.0),
 np.str_('literature'): np.float64(1.0),
 np.str_('psychology'): np.float64(0.0),
 np.str_('science_and_engineering'): np.float64(0.0)}

In [48]:
for alpha in [0.01, 0.1, 0.5, 1.0, 2.0]:
    m = MultinomialNB(alpha=alpha).fit(Xv_train, y_train)
    p = m.predict(Xv_test)
    print(f"alpha={alpha:<5} acc={accuracy_score(y_test, p):.4f}  macroF1={f1_score(y_test, p, average='macro'):.4f}")

alpha=0.01  acc=0.8109  macroF1=0.7063
alpha=0.1   acc=0.8119  macroF1=0.7155
alpha=0.5   acc=0.8167  macroF1=0.7200
alpha=1.0   acc=0.8192  macroF1=0.7143
alpha=2.0   acc=0.8084  macroF1=0.6688


In [49]:
yazarlar_df = df_clean[df_clean.book_author != "Kolektif"]
top_yazarlar = yazarlar_df.book_author.value_counts().head(10).index.tolist()
df_yazar = yazarlar_df[yazarlar_df.book_author.isin(top_yazarlar)].copy()
df_yazar.book_author.value_counts()

book_author
Sigmund Freud                    157
Tahir Özakkaş                    146
Sabahattin Ali                   109
Stefan Zweig                     100
Fyodor Mihayloviç Dostoyevski     99
Hüseyin Rahmi Gürpınar            99
Lev Nikolayeviç Tolstoy           95
Mutlu Dinç                        92
Jack London                       90
Prof. Dr. Bahri Öztürk            87
Name: count, dtype: int64

In [50]:
def yazar_adini_sil(row):
    parcalar = [p for p in clean(row["book_author"]).split() if len(p) >= 3 and p not in ("prof", "sir")]
    kelimeler = [k for k in row["clean"].split() if not any(k.startswith(p) for p in parcalar)]
    return " ".join(kelimeler)

df_yazar["clean_adsiz"] = df_yazar.apply(yazar_adini_sil, axis=1)
df_yazar[["book_author", "clean_adsiz"]].head()

,book_author,clean_adsiz
1293,Mutlu Dinç,libra mevzuat serimizin temel kitaplarından ol...
1840,Mutlu Dinç,sigortacılık sektörü türkiyede önemli bir geli...
3727,Tahir Özakkaş,bütüncül psikoterapi hastanın bilişlerinin dav...
3728,Tahir Özakkaş,bütüncül psikoterapi hastanın bilişlerinin dav...
3733,Tahir Özakkaş,bütüncül psikoterapi hastanın bilişlerinin dav...


In [51]:
Xa = df_yazar.clean_adsiz.to_numpy()
ya = df_yazar.book_author.to_numpy()
Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.2, random_state=42, stratify=ya)

vec_a = CountVectorizer(min_df=2)
Xav_train = vec_a.fit_transform(Xa_train)
Xav_test = vec_a.transform(Xa_test)

model_a = MultinomialNB().fit(Xav_train, ya_train)
pa = model_a.predict(Xav_test)
print("Test accuracy:", accuracy_score(ya_test, pa))
print("Test macro F1:", f1_score(ya_test, pa, average="macro"))
print(classification_report(ya_test, pa))

Test accuracy: 0.8697674418604651
Test macro F1: 0.861306682518939
                               precision    recall  f1-score   support

Fyodor Mihayloviç Dostoyevski       0.68      0.85      0.76        20
       Hüseyin Rahmi Gürpınar       0.95      0.95      0.95        20
                  Jack London       0.92      0.67      0.77        18
      Lev Nikolayeviç Tolstoy       0.92      0.58      0.71        19
                   Mutlu Dinç       1.00      0.94      0.97        18
       Prof. Dr. Bahri Öztürk       0.89      1.00      0.94        17
               Sabahattin Ali       0.81      1.00      0.90        22
                Sigmund Freud       0.86      0.94      0.90        32
                 Stefan Zweig       0.75      0.75      0.75        20
                Tahir Özakkaş       1.00      0.93      0.96        29

                     accuracy                           0.87       215
                    macro avg       0.88      0.86      0.86       215
        